<a href="https://colab.research.google.com/github/pajola/secai-course/blob/main/module-02-data-security/01_inventory_and_pii.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" width="150"></a>

# 1 · Inventory and PII discovery — what is in this dataset, and who can be identified from it

**SecAI course · Data Security for AI, part 1**

**Scenario.** Northvale Lending (fictional) wants to train a loan-approval model on 1,200 historical applications.
Before anything reaches a training pipeline, the AI security team runs a **data risk review**. You are the reviewer who signs off on the dataset, or blocks it.
This notebook covers the first half of the review: *what is in the data* and *which of it is personal*.
Notebooks 2 and 3 cover bias and provenance; notebook 4 puts everything into one report.

A dataset is an attack surface: once it is used for training, whatever personal data it contains can leak, through a breach of the
training set itself or through the model (memorisation, membership inference — module 1, notebook 4).

| | Threat to the data |
|---|---|
| **Goal** | learn personal or sensitive facts about the people in the dataset (identity, income, health, bank details) |
| **Knowledge** | the released dataset, or a model trained on it, plus **public auxiliary data** (social media, voter lists, data breaches) |
| **Capability** | read access: a leaked training set, a "de-identified" dataset shared with a partner, an insider, or queries to the trained model |

All data is synthetic and invented (company, people, IBANs, notes), generated by [`northvale.py`](northvale.py) with seed 42.

**Open-source tools used here:** [Microsoft Presidio](https://microsoft.github.io/presidio/) (PII detection and redaction)
and [python-stdnum](https://arthurdejong.org/python-stdnum/) (IBAN validation).
Each check is first written by hand, then compared with the tool.

In [1]:
# Setup — on Google Colab, fetch the dataset generator and install the open-source tools used below (about a minute).
import os, sys, importlib.util, urllib.request
if "google.colab" in sys.modules:
    if not os.path.exists("northvale.py"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pajola/secai-course/main/module-02-data-security/northvale.py", "northvale.py")
    %pip install -q faker presidio-analyzer presidio-anonymizer python-stdnum
    if importlib.util.find_spec("en_core_web_sm") is None:
        !python -m spacy download en_core_web_sm

In [2]:
import re, numpy as np, pandas as pd
from northvale import load, REF_DATE
pd.set_option("display.max_colwidth", 90)

df, manifest = load()
print(f"{len(df)} rows x {df.shape[1]} columns")
df.head(3)

1200 rows x 13 columns


,applicant_id,full_name,email,phone,date_of_birth,gender,postcode,annual_income_eur,credit_score,loan_amount_eur,advisor_notes,source_batch,approved
0,NV-10627,Costanzo Asmundo,costanzo.asmundo33@fastwebnet.it,+39 0444218190,1967-11-28,M,35136,10300.0,618,4500.0,"Existing customer, no missed payments. Passport scan saved in the shared folder.",B3_broker_purchase,0
1,NV-10559,Beatrice Galeati,beatrice.galeati93@tiscali.it,+39 0923386370,1981-12-30,F,35135,25700.0,714,18500.0,Wants to consolidate two existing loans.,B3_broker_purchase,0
2,NV-11185,Piersanti Morpurgo,piersanti.morpurgo57@fastwebnet.it,+39 04345116150,1974-12-18,M,35121,31100.0,601,18000.0,"Fixed-term contract, renewal expected.",B2_core_banking,0


## 1.1 Inventory

You can't protect what you haven't inventoried. For each column: its type, how many distinct values it has, whether it has missing values, and one example.

In [3]:
inventory = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "distinct": df.nunique(),
    "missing": df.isna().sum(),
    "example": df.iloc[0],
})
inventory

,dtype,distinct,missing,example
applicant_id,object,1200,0,NV-10627
full_name,object,1198,0,Costanzo Asmundo
email,object,1200,0,costanzo.asmundo33@fastwebnet.it
phone,object,1200,0,+39 0444218190
date_of_birth,object,1144,0,1967-11-28
gender,object,2,0,M
postcode,object,8,0,35136
annual_income_eur,float64,440,0,10300.0
credit_score,int64,300,0,618
loan_amount_eur,float64,115,0,4500.0


`applicant_id`, `full_name`, `email` and `phone` are distinct for (almost) every row: they point at one person. `advisor_notes` is free text,
so its type (`object`) tells you nothing about what is inside. That is where we look next.

## 1.2 Structured PII: scan every column with patterns

Column names can lie (a column called `ref` may hold tax codes), so a scan looks at the **values**, not the names.
A few regular expressions, applied to every text column: what fraction of the values match each pattern?

In [4]:
PATTERNS = {
    "email":   r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone":   r"\+?\d[\d ]{7,14}\d",
    "date":    r"\b\d{4}-\d{2}-\d{2}\b",
    "iban":    r"\b[A-Z]{2}\d{2}(?: ?[A-Z0-9]){11,30}\b",
}
text_cols = df.select_dtypes("object").columns
scan = pd.DataFrame({name: [df[c].astype(str).str.contains(p).mean() for c in text_cols] for name, p in PATTERNS.items()},
                    index=text_cols)
scan.style.format("{:.1%}").background_gradient(vmin=0, vmax=1, cmap="Reds")

,email,phone,date,iban
applicant_id,0.0%,0.0%,0.0%,0.0%
full_name,0.0%,0.0%,0.0%,0.0%
email,100.0%,0.0%,0.0%,0.0%
phone,0.0%,100.0%,0.0%,0.0%
date_of_birth,0.0%,0.0%,100.0%,0.0%
gender,0.0%,0.0%,0.0%,0.0%
postcode,0.0%,0.0%,0.0%,0.0%
advisor_notes,0.0%,10.6%,0.0%,10.6%
source_batch,0.0%,0.0%,0.0%,0.0%


The structured columns light up as expected: `email`, `phone`, `date_of_birth` (100%). The surprise is `advisor_notes`: about 10% of the notes contain
something that looks like an IBAN. The phone pattern also fires on some notes; those turn out to be the IBAN digits — a pattern is only a candidate.

## 1.3 Free text: where PII hides

Before writing any detector, **read a random sample** of the free text. The keywords below come from doing exactly that.

In [5]:
for note in df.advisor_notes.sample(8, random_state=1):
    print("•", note)

• Stable employment, 8 years at current employer. Salary credited to IBAN IT19X6019134438998278020586.
• First-time customer, referred by a branch colleague. Salary credited to IBAN IT08C3717850902556184442182.
• Loan for car purchase, quote attached. Co-signer: Damiano Bruno, spouse.
• Permanent contract, public sector. Salary credited to IBAN IT97N9984802205093869484249.
• Wants to consolidate two existing loans.
• Loan for car purchase, quote attached. Salary credited to IBAN IT83L0982210401648188044287. Diabetic, regular check-ups, no impact on job.
• Self-employed, income documented via tax returns.
• Permanent contract, public sector. Copy of ID card scanned and attached to the file.


### IBANs: a regex finds candidates, a checksum confirms them

An IBAN has two check digits (positions 3–4) computed with mod-97 over the whole number (ISO 13616). A random string of the right shape
passes with probability 1/97, so the checksum separates real account numbers from typos and reference numbers.

In [6]:
IBAN_RE = re.compile(r"\bIT\d{2}(?: ?[A-Z0-9]){23}\b")      # Italian IBAN: IT + 2 check digits + 23 characters

def iban_valid(s):
    s = s.replace(" ", "")
    s = s[4:] + s[:4]                                    # move country code and check digits to the end
    return int("".join(str(int(c, 36)) for c in s)) % 97 == 1   # letters -> numbers (A=10 ... Z=35), then mod 97

candidates = df.advisor_notes.str.findall(IBAN_RE)
has_candidate = candidates.str.len() > 0
has_iban = candidates.apply(lambda found: any(iban_valid(x) for x in found))
print(f"rows with an IBAN-shaped string: {has_candidate.sum()}")
print(f"rows with a valid IBAN:          {has_iban.sum()}  ({has_iban.mean():.1%})")
print("\nshaped like an IBAN, but the checksum fails:")
for found in candidates[has_candidate & ~has_iban].head(3): print("  ", found)

rows with an IBAN-shaped string: 127
rows with a valid IBAN:          110  (9.2%)

shaped like an IBAN, but the checksum fails:
   ['IT92F7681722577545847262906']
   ['IT06L5078468509574430349180']
   ['IT65W1410527698198780927046']


In [7]:
from stdnum import iban as stdnum_iban     # the same check, from a maintained library (handles every country's format)
print("python-stdnum agrees on every row:",
      (candidates.apply(lambda found: any(stdnum_iban.is_valid(x) for x in found)) == has_iban).all())

python-stdnum agrees on every row: True


### Special-category data and identity documents

GDPR Art. 9 lists **special categories** (health, ethnic origin, religion, sexual orientation, ...) that need an explicit legal basis even to be stored.
A loan advisor writing *"recent hospital stay"* in a notes field has turned a credit dataset into a health dataset.

In [8]:
FREE_TEXT_RULES = {
    "IBAN (valid checksum)":    None,       # computed above
    "health (GDPR Art. 9)":     r"hospital\w*|sick leave|chemotherapy|diabetic|pregnan\w*|medication|depression|surgery",
    "ID document scan":         r"ID card|passport|driving licence",
    "third-party name":         r"Co-signer: ",
}
flags = pd.DataFrame({k: (has_iban if p is None else df.advisor_notes.str.contains(p, case=False)) for k, p in FREE_TEXT_RULES.items()})
summary = pd.DataFrame({"rows": flags.sum(), "share": flags.mean().map("{:.1%}".format)})
print(summary, f"\n\nrows with at least one finding: {flags.any(axis=1).sum()} ({flags.any(axis=1).mean():.1%})")

                       rows share
IBAN (valid checksum)   110  9.2%
health (GDPR Art. 9)     97  8.1%
ID document scan         86  7.2%
third-party name         60  5.0% 

rows with at least one finding: 315 (26.2%)


The structured PII was obvious; this is the real finding. 110 IBANs (9.2%), 97 health mentions (8.1%), 86 references to ID-document scans (7.2%),
and 60 names of people who never applied for anything (co-signers), all in a field nobody listed as personal data.

Free text is also the field a model is most likely to **memorise and regurgitate**: a language model trained on these notes can complete
*"Salary credited to IBAN IT…"* with a real account number (Carlini et al., *The Secret Sharer*, 2019).

## 1.4 The same job with Microsoft Presidio

Presidio combines regular expressions, checksums, context words and a spaCy named-entity model.
We use the small English spaCy model (`en_core_web_sm`) to keep the install light.

In [9]:
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider

nlp = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy", "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}]}).create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp, supported_languages=["en"])

def presidio_rows(entities, threshold=0.5):
    """For each entity type: which rows of advisor_notes contain at least one detection."""
    hits = pd.DataFrame(False, index=df.index, columns=entities)
    for i, note in df.advisor_notes.items():
        for r in analyzer.analyze(note, language="en", entities=entities, score_threshold=threshold):
            hits.loc[i, r.entity_type] = True
    return hits

ENTITIES = ["IBAN_CODE", "PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"]
found = presidio_rows(ENTITIES)
print(found.sum().to_string())
print("\nIBAN rows identical to our checksum detector:", (found.IBAN_CODE == has_iban).all())

IBAN_CODE        110
PERSON           158
EMAIL_ADDRESS      0
PHONE_NUMBER       0

IBAN rows identical to our checksum detector: True


Presidio's IBAN recogniser also validates the checksum, so it finds exactly our 110 rows and ignores the decoys.
For people, it relies on the NER model. How does it compare with the 60 co-signer names?

In [10]:
person = found.PERSON
cosigner = flags["third-party name"]
print(f"PERSON rows: {person.sum()}  ->  co-signer rows caught: {(person & cosigner).sum()}/{cosigner.sum()},"
      f"  flagged rows without a co-signer: {(person & ~cosigner).sum()}")
print("\nexamples of false alarms:")
for i in df.index[person & ~cosigner][:3]:
    hit = [note for note in analyzer.analyze(df.advisor_notes[i], language="en", entities=["PERSON"], score_threshold=0.5)]
    print(f"  {df.advisor_notes[i]!r}\n     -> PERSON = {[df.advisor_notes[i][h.start:h.end] for h in hit]}")

PERSON rows: 158  ->  co-signer rows caught: 51/60,  flagged rows without a co-signer: 107

examples of false alarms:
  'Wants to consolidate two existing loans. Salary credited to IBAN IT21 H633 7922 3410 3469 9673 311.'
     -> PERSON = ['Salary']
  'Permanent contract, public sector. Salary credited to IBAN IT97 S507 3214 6449 3699 9516 793. Pregnant, maternity leave expected in the autumn.'
     -> PERSON = ['Salary']
  'Loan for home renovation, contractor estimate provided. Salary credited to IBAN IT38O2426423642813190924876.'
     -> PERSON = ['Salary']


The small NER model catches most co-signers, but also flags ordinary words such as "Salary" as people. A detector has a precision and a recall like any classifier:
measure both on a hand-labelled sample before trusting the counts. A larger model (`en_core_web_lg`, ~600 MB) reduces the false alarms.

What about health and identity documents? Presidio has **no built-in recogniser** for either, because they are domain-specific. We add our own:

In [11]:
analyzer.registry.add_recognizer(PatternRecognizer(supported_entity="HEALTH",
    patterns=[Pattern("health", FREE_TEXT_RULES["health (GDPR Art. 9)"], 0.6)]))
analyzer.registry.add_recognizer(PatternRecognizer(supported_entity="ID_DOCUMENT",
    patterns=[Pattern("id_doc", FREE_TEXT_RULES["ID document scan"], 0.6)]))

ENTITIES = ["IBAN_CODE", "PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "HEALTH", "ID_DOCUMENT"]
found = presidio_rows(ENTITIES)
print(found.sum().to_string())

IBAN_CODE        110
PERSON           158
EMAIL_ADDRESS      0
PHONE_NUMBER       0
HEALTH            97
ID_DOCUMENT       86


Same counts as our hand-written rules (97 and 86), and now everything runs through one engine that can also **redact**.
The lesson matches the one from PhantomText in module 1: a detector only finds what its rules or training data describe. The defaults miss the categories
that matter most here (health), so a review always starts by reading the data.

## 1.5 Classification and minimisation

Every column gets a label and an action. **Minimisation first**: does the model need this field at all? Only what stays gets masked or generalised.

| Field | Label | Action |
|---|---|---|
| `applicant_id` | Internal | Keep (pseudonymous key) |
| `full_name`, `email`, `phone` | Confidential / PII | **Remove**: no predictive value |
| `date_of_birth` | Confidential / PII | **Generalise** to an age band |
| `gender` | Sensitive / protected attribute | **Exclude** from features; keep in a separate audit table for fairness testing (notebook 2) |
| `postcode` | Confidential / quasi-identifier | Keep for now, flag as a proxy risk (notebook 2) |
| `annual_income_eur`, `credit_score`, `loan_amount_eur` | Confidential | Keep |
| `advisor_notes` | **Restricted** (IBAN, health) | Redact, or drop the field |
| `approved` | Internal | Label; check for historical bias (notebook 2) |

If the notes had to stay (for example to train a model that reads them), Presidio's anonymiser replaces each detection with a placeholder:

In [12]:
from presidio_anonymizer import AnonymizerEngine
anonymizer = AnonymizerEngine()

def redact(note):
    return anonymizer.anonymize(text=note, analyzer_results=analyzer.analyze(note, language="en", entities=ENTITIES, score_threshold=0.5)).text

for i in df.index[flags.sum(axis=1) >= 2][:4]:
    print("before:", df.advisor_notes[i]); print("after: ", redact(df.advisor_notes[i]), "\n")

before: Loan for home renovation, contractor estimate provided. Photo of driving licence received by WhatsApp. Co-signer: Alderano Lettiere, spouse.
after:  Loan for home renovation, contractor estimate provided. Photo of <ID_DOCUMENT> received by WhatsApp. Co-signer: <PERSON>, spouse. 

before: Permanent contract, public sector. Salary credited to IBAN IT97 S507 3214 6449 3699 9516 793. Pregnant, maternity leave expected in the autumn.
after:  Permanent contract, public sector. <PERSON> credited to IBAN <IBAN_CODE>. <HEALTH>, maternity leave expected in the autumn. 

before: First-time customer, referred by a branch colleague. Salary credited to IBAN IT22G1268812282178192365346. Passport scan saved in the shared folder.
after:  First-time customer, referred by a branch colleague. <PERSON> credited to IBAN <IBAN_CODE>. <ID_DOCUMENT> scan saved in the shared folder. 

before: Fixed-term contract, renewal expected. Was on sick leave for three months last year (back injury). Co-signer: Do

Redaction keeps the rest of the sentence, including whatever the detector missed, and it inherits the detector's false alarms ("Salary" became `<PERSON>`). Here no model feature needs the notes, so the safer choice is to **drop the field**.

In [13]:
def age_years(dob):
    return ((REF_DATE - pd.to_datetime(dob)).dt.days // 365.25).astype(int)

def age_band(age):
    return pd.cut(age, [0, 30, 40, 50, 60, 200], right=False, labels=["<30", "30-39", "40-49", "50-59", "60+"])

def minimise(df):
    out = df.drop(columns=["full_name", "email", "phone", "advisor_notes", "gender"])
    out["age_band"] = age_band(age_years(out.pop("date_of_birth")))
    return out

audit_table = df[["applicant_id", "gender"]]          # kept apart, access-controlled, used only for fairness testing
train_ready = minimise(df)
train_ready.head()

,applicant_id,postcode,annual_income_eur,credit_score,loan_amount_eur,source_batch,approved,age_band
0,NV-10627,35136,10300.0,618,4500.0,B3_broker_purchase,0,50-59
1,NV-10559,35135,25700.0,714,18500.0,B3_broker_purchase,0,40-49
2,NV-11185,35121,31100.0,601,18000.0,B2_core_banking,0,50-59
3,NV-10244,35124,40800.0,678,24500.0,B2_core_banking,0,<30
4,NV-10477,35121,31000.0,779,8500.0,B1_core_banking,1,50-59


## 1.6 Attack the defense: is the minimised data anonymous?

Names are gone. But **quasi-identifiers**, attributes that are not identifying alone, single people out in combination.
Sweeney (2002) showed that date of birth + ZIP code + sex identify about 87% of the US population.

The attacker's tool is a **linkage attack**: join the released table with a public list that has names. Suppose the bank had released the data with names removed
but the exact date of birth kept, and the attacker scraped 50 profiles (name, birthday, postcode, gender) from social media:

In [14]:
public = df.sample(50, random_state=7)[["full_name", "date_of_birth", "postcode", "gender"]]       # what the attacker knows
released = df.drop(columns=["full_name", "email", "phone", "advisor_notes"])                       # "de-identified": no names

linked = public.merge(released, on=["date_of_birth", "postcode", "gender"])
matches = linked.groupby("full_name").size()
print(f"people in the attacker's list: {len(public)},  re-identified with a unique match: {(matches == 1).sum()}")
linked[["full_name", "annual_income_eur", "credit_score", "loan_amount_eur", "approved"]].head(4)

people in the attacker's list: 50,  re-identified with a unique match: 50


,full_name,annual_income_eur,credit_score,loan_amount_eur,approved
0,Michele Mazzanti,34900.0,622,7000.0,0
1,Maurizio Vianello,27200.0,617,25500.0,0
2,Cristina Lettiere,42500.0,795,14500.0,1
3,Guglielmo Cavalcanti,24700.0,552,26500.0,1


Every one of them is re-identified, with income, credit score and the loan decision attached. Removing names is **pseudonymisation**, not anonymisation:
under GDPR the data is still personal data.

How much does generalisation help? For each choice of quasi-identifiers, the share of rows that are **unique** (an equivalence class of size 1),
and **k**, the size of the smallest class: a table is *k-anonymous* if every row is indistinguishable from at least k−1 others.

In [15]:
age = age_years(df.date_of_birth)
LEVELS = {
    "date of birth + postcode + gender":  pd.DataFrame({"dob": df.date_of_birth, "pc": df.postcode, "g": df.gender}),
    "age (years) + postcode + gender":    pd.DataFrame({"age": age, "pc": df.postcode, "g": df.gender}),
    "age band + postcode + gender":       pd.DataFrame({"band": age_band(age), "pc": df.postcode, "g": df.gender}),
    "age band + postcode (our release)":  pd.DataFrame({"band": age_band(age), "pc": df.postcode}),
    "age band only":                      pd.DataFrame({"band": age_band(age)}),
}
def class_sizes(qi):
    key = qi.astype(str).agg("|".join, axis=1)
    return key.map(key.value_counts())

rows = []
for name, qi in LEVELS.items():
    size = class_sizes(qi)
    rows.append({"quasi-identifiers": name, "unique rows": f"{(size == 1).mean():.1%}", "k": size.min(),
                 "rows in classes < 5": f"{(size < 5).mean():.1%}"})
pd.DataFrame(rows).set_index("quasi-identifiers")

,unique rows,k,rows in classes < 5
quasi-identifiers,,,
date of birth + postcode + gender,99.2%,1,100.0%
age (years) + postcode + gender,17.5%,1,79.1%
age band + postcode + gender,0.2%,1,3.2%
age band + postcode (our release),0.0%,2,1.0%
age band only,0.0%,44,0.0%


Generalising the date of birth to an age band and moving gender to the audit table removes most of the risk, but **k is still small**:
the 60+ applicants in some postcodes form tiny groups. Small groups are where re-identification happens, and the same 60+ group shows up again
in notebook 2 as a representation problem.

k-anonymity is not enough on its own. **l-diversity** asks: within each group, how many distinct values of the sensitive attribute (here the loan decision) appear?
With l = 1, everyone in some group has the same outcome, so an attacker learns the decision without knowing which row is the target.

In [16]:
QI = ["age_band", "postcode"]
groups = train_ready.groupby(QI, observed=True).approved.agg(size="size", distinct_outcomes="nunique", approval_rate="mean")
print("k-anonymity:", groups["size"].min())
print("l-diversity (approved):", groups["distinct_outcomes"].min())
groups[groups["size"] < 5]

k-anonymity: 2
l-diversity (approved): 1


size  distinct_outcomes  approval_rate
age_band postcode                                        
60+      35121        3                  2       0.666667
         35122        2                  2       0.500000
         35123        4                  2       0.750000
         35131        3                  1       1.000000

Dedicated tools such as [ARX](https://arx.deidentifier.org) or [pycanon](https://github.com/IFCA-Advanced-Computing/pycanon) compute these metrics and stronger ones
(t-closeness, δ-presence). pycanon is not used here: on Colab it replaces the preinstalled pandas and scipy, which can hang the running session.

To reach a target such as k ≥ 5, merge the small classes (e.g. one band "50+"), suppress the few rows that still stand out, or move to synthetic data or
differential privacy when the data has to leave the organisation. Every step trades utility for privacy, and the right trade-off depends on who receives the data.

## Take-aways

* **Inventory first.** Scan values, not column names; read a sample of every free-text field.
* **Free text is where PII hides**: bank details, health data, ID documents, third parties who never consented. It is also what models memorise.
* **Validate, don't just match**: a checksum turns 127 IBAN-shaped strings into 110 real ones. Tools such as Presidio do this for standard types, but have **no rules for domain-specific categories** (health) until you add them.
* **Minimise before you mask**: drop what the model doesn't need; generalise what it does.
* **Removing names is not anonymisation.** Quasi-identifiers enable linkage attacks; measure k-anonymity and l-diversity, and watch the small groups.

### References
* Sweeney, *k-Anonymity: A Model for Protecting Privacy*, IJUFKS 2002.
* Machanavajjhala et al., *l-Diversity: Privacy Beyond k-Anonymity*, ACM TKDD 2007.
* Narayanan & Shmatikov, *Robust De-anonymization of Large Sparse Datasets* (Netflix Prize), IEEE S&P 2008.
* Carlini et al., *The Secret Sharer: Evaluating and Testing Unintended Memorization in Neural Networks*, USENIX Security 2019.
* Regulation (EU) 2016/679 (GDPR), Art. 4(1), 4(5) (pseudonymisation), 5(1)(c) (data minimisation), 9 (special categories).
* Microsoft Presidio documentation · ISO 13616 (IBAN) · Prasser et al., *ARX — A Comprehensive Tool for Anonymizing Biomedical Data*, AMIA 2014.